# Preliminary Colab adaptation: collaborating small and large models

**Original paper:** Peng Chen and Bang Wang, *On Collaborating Small and Large Models For Few-shot Intent Detection*, Findings of EMNLP 2025, pp. 13937–13953.

## Honest status
The earlier two-linear-classifier experiment is retained only as a **preliminary concept demonstration** in `results.json`; it is not an implementation of FCSLM. This notebook now defines a more faithful free-Colab path: a small few-shot classifier proposes candidate intents and confidence, then an instruction-tuned public LLM receives the candidate descriptions and decides whether to retain or revise the intent. OOS detection and LLM self-supervised augmentation are not implemented in this small path. The model cells are **not executed locally** because `torch` and `transformers` are unavailable here.

In [ ]:
!pip -q install "transformers>=4.45" "datasets>=2.20" accelerate sentencepiece scikit-learn

In [ ]:
import csv, io, json, random, urllib.request, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from transformers import pipeline

SEED=42; random.seed(SEED); np.random.seed(SEED)
URL="https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
TEST="https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
def load(url): return list(csv.DictReader(io.StringIO(urllib.request.urlopen(url).read().decode())))
rows=load(URL); test=load(TEST); classes=sorted({r["category"] for r in rows})[:10]
rows=[r for r in rows if r["category"] in classes]; test=[r for r in test if r["category"] in classes]
X=np.array([r["text"] for r in rows]); y=np.array([r["category"] for r in rows]); Xt=np.array([r["text"] for r in test]); yt=np.array([r["category"] for r in test])
Xtr,_,ytr,_=train_test_split(X,y,test_size=.75,random_state=SEED,stratify=y)
vec=TfidfVectorizer(ngram_range=(1,2)); Z=vec.fit_transform(Xtr); clf=LogisticRegression(max_iter=400,random_state=SEED).fit(Z,ytr)
small_pred=clf.predict(vec.transform(Xt)); small_conf=clf.predict_proba(vec.transform(Xt)).max(1)
print("few-shot examples",len(Xtr),"classes",classes)

In [ ]:
# Public small instruction model as the large-model proxy. This is central-model routing, not a reproduction of the paper.
llm=pipeline("text2text-generation",model="google/flan-t5-small",device=-1)
label_text="; ".join(classes)
def ask_llm(text,candidate):
    prompt=f"Classify the banking request into exactly one label from: {label_text}. Candidate from a small model: {candidate}. Request: {text}. Answer with one label only."
    answer=llm(prompt,max_new_tokens=8,do_sample=False)[0]["generated_text"].strip()
    return answer if answer in classes else candidate

# Keep the run small enough for free Colab; report the routed count and exact sample cap.
limit=min(100,len(Xt)); routed=[]
for text,cand,conf in zip(Xt[:limit],small_pred[:limit],small_conf[:limit]):
    routed.append(ask_llm(text,cand) if conf<0.80 else cand)
print("routed examples",sum(a!=b for a,b in zip(routed,small_pred[:limit])),"of",limit)
# Add an evaluation cell after a successful run; do not infer metrics from this setup cell.

## What remains unverified

This notebook now captures the paper’s central inference pattern—small-model candidate generation followed by an LLM refinement step—but it is not a completed replication. It does not implement the paper’s self-supervised LLM augmentation, multi-round reasoning, enriched intent descriptions, or OOS detection. Run all cells in Colab, add held-out accuracy/F1 and routed-fraction outputs, and save them as a new verified artifact before calling this an implementation.